In [0]:
#🧱 Part 1: Data Ingestion and Preparation
#1. Load the Data

from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import lit
from pyspark.sql.functions import col

# Infer schema from one file, then set all columns to StringType
sample_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/assignment_3_listings/austin_listings.csv")
)
columns = sample_df.columns
schema = StructType([StructField(col, StringType(), True) for col in columns])

#Austin
df_austin = (spark.read
    .option("header", True)
    .schema(schema)
    .csv("/Volumes/workspace/default/assignment_3_listings/austin_listings.csv")
    .withColumn("city", lit("Austin"))
)
display(df_austin)

#Bangkok
df_bangkok = (spark.read
    .option("header", True)
    .schema(schema)
    .csv("/Volumes/workspace/default/assignment_3_listings/bangkok_listings.csv")
    .withColumn("city", lit("Bangkok"))
)

#Buenos Aires
df_buenos_aires = (spark.read
    .option("header", True)
    .schema(schema)
    .csv("/Volumes/workspace/default/assignment_3_listings/buenos_aires_listings.csv")
    .withColumn("city", lit("Buenos Aires"))
)

#Cape town
df_cape_town = (spark.read
    .option("header", True)
    .schema(schema)
    .csv("/Volumes/workspace/default/assignment_3_listings/cape_town_listings.csv")
    .withColumn("city", lit("Cape Town"))
)

#Istanbul
df_istanbul = (spark.read
    .option("header", True)
    .schema(schema)
    .csv("/Volumes/workspace/default/assignment_3_listings/istanbul_listings.csv")
    .withColumn("city", lit("Istanbul"))
)

#Melbourne
df_melbourne = (spark.read
    .option("header", True)
    .schema(schema)
    .csv("/Volumes/workspace/default/assignment_3_listings/melbourne_listings.csv")
    .withColumn("city", lit("Melbourne"))
)

#2. Merge Datasets
# Union all DataFrames
df_city_files = (
    df_austin
    .unionByName(df_bangkok)
    .unionByName(df_buenos_aires)
    .unionByName(df_cape_town)
    .unionByName(df_istanbul)
    .unionByName(df_melbourne)
)

#3. Initial Exploration
#Print the number of rows per city.
df_city_files.groupBy("city").count().show()
        
#Display the first 5 rows of the combined dataset.
df_city_files.show(5)

#Use .describe() to show basic statistics for: price, minimum_nights, number_of_reviews
display(df_city_files.describe("price", "minimum_nights", "number_of_reviews"))

#4. Clean the Data
#Convert the price column to numeric (strip dollar signs, commas, etc.).
from pyspark.sql.functions import regexp_replace, col, to_date
df_city_clean = df_city_files.withColumn(
    "price",
    regexp_replace(col("price"), "[$,]", "").cast("double")
)

#Convert last_review to datetime.
df_city_clean = df_city_clean.withColumn(
    "last_review",
    to_date(col("last_review"), "yyyy-MM-dd")
)

#Fill missing values: Fill reviews_per_month with 0, Fill last_review with NaT or similar placeholder
df_city_clean = (df_city_clean
    .fillna({"reviews_per_month": 0}) 
    .fillna({"last_review": 'NaT'})      
)

#Drop rows that have missing values in: host_id,neighbourhood,room_type
df_city_clean = df_city_clean.dropna(subset=["host_id", "neighbourhood", "room_type"])
df_city_clean.select("host_id", "neighbourhood", "room_type").show(5)


In [0]:
%fs ls dbfs:/Volumes/workspace/default/assignment_3_listings

In [0]:
#📊 Part 2: Analysis & Engineering
#5. Top Hosts by Number of Listings: Identify the top 10 hosts (by host_id) across all cities who have the most listings.
from pyspark.sql.functions import col
df_top_hosts = df_city_clean.groupBy("host_id").count().orderBy(col("count").desc()).limit(10)
display(df_top_hosts)

#6. Price Distribution by City: For each city, calculate: Average price, Median price, Minimum and maximum price
from pyspark.sql.functions import avg, min, max, col, expr
df_price_dist = df_city_clean.groupBy("city").agg(
        avg(col("price")).alias("avg_price"),
        expr("percentile_approx(price, 0.5)").alias("median_price"),
        min(col("price")).alias("min_price"),
        max(col("price")).alias("max_price")
    )
display(df_price_dist)

#7. Availability Analysis
#Find the percentage of listings with availability_365 > 300 days for each city.
from pyspark.sql.functions import avg, col, expr

df_city_avail_365 = df_city_clean.groupBy("city").agg(
    (
        avg(
            expr("try_cast(availability_365 as int) > 300")
            .cast("int")
        ) * 100
    ).alias("pct_avail_365_gt_300"),
    avg(expr("try_cast(availability_365 as int)")).alias("avg_availability_365")
)
display(df_city_avail_365)

#Calculate the average availability per city.
df_city_avg_avail = df_city_clean.groupBy("city").agg(
    avg(expr("try_cast(availability_365 as int)")).alias("avg_availability_365")
)
display(df_city_avg_avail)

#8. Neighborhood Insights
#For each city: Find the top 3 neighborhoods with the most listings.
from pyspark.sql.functions import col, collect_list, struct, count
df_city_neighborhoods = df_city_clean.groupBy("city", "neighbourhood").count().orderBy(col("count").desc()).groupBy("neighbourhood").agg(collect_list(struct("count")).alias("top_neighborhoods")).show(3)
display(df_city_neighborhoods)

#For each top neighborhood: Calculate average price, Calculate average number of review, Identify the most common room type
df_city_neighbourhoods_avg = df_city_clean.groupBy("city", "neighbourhood").agg(
        avg(expr("try_cast(price as double)")).alias("avg_price"),
        avg(expr("try_cast(number_of_reviews as int)")).alias("avg_number_of_reviews"),
        expr("mode(room_type)").alias("most_common_room_type"))
display(df_city_neighbourhoods_avg)

#9. Room Type Trends
#Group by room_type and calculate: Average price, Number of listings, Average availability
df_city_room_type = df_city_clean.groupBy("city", "room_type").agg(avg(expr("try_cast(price as double)")).alias("avg_price"), count("*").alias("num_listings"), avg(expr("try_cast(availability_365 as int)")).alias("avg_availability_365"))
display(df_city_room_type)

#10. Review Trends
#Calculate the average number of reviews per listing, per city.
df_city_reviews = df_city_clean.groupBy("city").agg(avg(expr("try_cast(number_of_reviews as int)")).alias("avg_number_of_reviews"))
display(df_city_reviews)                                       

#Identify the top 5 listings (by id) with the highest number of reviews across all cities.
df_city_top_reviews = df_city_clean.select("id", "city", "name", "host_id", "number_of_reviews", "price").orderBy(col("number_of_reviews").desc()).limit(5)
display(df_city_top_reviews)

In [0]:
price_neigh_room = (
    df_city_clean
    .groupBy("city", "neighbourhood", "room_type")
    .agg(avg("price").alias("avg_price"))
    .orderBy("city", col("avg_price").desc())
)

display(price_neigh_room)